# 02. PE label and prior audit

Audit the released PE labels and exact PE-prior information needed for population reweighting. Start with `GW230627_015337`; do not choose labels using ESS or the eventual population score.


In [1]:
from pathlib import Path
import json
import os

import h5py
import numpy as np
import pandas as pd

from astropy.constants import c
from bilby.gw.cosmology import get_cosmology
from bilby.gw.prior import (
    UniformInComponentsChirpMass,
    UniformInComponentsMassRatio,
    UniformSourceFrame,
)
import bilby
from bilby.gw.cosmology import get_cosmology
from astropy.constants import c

EVENT_ID = "GW230627_015337"
inventory_dir = Path(os.environ["GWTC_RESULTS_ROOT"]) / "pilot" / "inventory"
index = json.loads((inventory_dir / "pe_release_index.json").read_text())
assert index["ready_for_label_audit"]


## 1. Released candidate labels

Enumerate the available results before applying any label-selection rule.


In [2]:
candidate_rows = []
for release, payload in index["releases"].items():
    if not payload.get("present", False):
        continue
    for event_id, event in payload["events"].items():
        for row in event["rows"]:
            candidate_rows.append({"release": release, "event_id": event_id, **row})

candidates = pd.DataFrame(candidate_rows)
event_candidates = candidates[candidates["event_id"] == EVENT_ID].copy()

event_candidates[[
    "release",
    "event_id",
    "result_samples_key",
    "mass_1_source_median",
    "mass_2_source_median",
    "redshift_median",
]]


,release,event_id,result_samples_key,mass_1_source_median,mass_2_source_median,redshift_median
0,o4a,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,8.483478,5.714881,0.066302
1,o4a,GW230627_015337,C00:Mixed,8.449413,5.742252,0.065766
2,o4a,GW230627_015337,C00:SEOBNRv5PHM,8.422318,5.762235,0.065132


## 2. Deep PESummary audit

Confirm sample counts and availability of the coordinates needed for PE-prior reconstruction.


In [3]:
deep_path = inventory_dir / f"pe_{EVENT_ID}_deep.json"
deep_report = json.loads(deep_path.read_text())

required = {
    "mass_1_source", "mass_2_source", "mass_ratio", "redshift",
    "luminosity_distance", "mass_1", "mass_2",
    "a_1", "a_2", "tilt_1", "tilt_2", "chi_eff", "chi_p",
}

rows = []
for analysis in deep_report["analyses"]:
    params = set(analysis["parameters"])
    rows.append({
        "label": analysis["label"],
        "sample_count": analysis["sample_count"],
        "parameter_count": len(params),
        "all_required_coordinates": required <= params,
    })

pd.DataFrame(rows)


,label,sample_count,parameter_count,all_required_coordinates
0,C00:IMRPhenomXPHM-SpinTaylor,16191,149,True
1,C00:Mixed,32382,149,True
2,C00:SEOBNRv5PHM,16577,149,True


## 3. Analysis metadata and mixed-posterior composition

Inspect only the metadata needed to understand the candidate labels.


In [4]:
shallow_report = json.loads(
    (inventory_dir / f"pe_{EVENT_ID}_shallow.json").read_text()
)
pe_path = Path(shallow_report["path"])
labels = [
    "C00:IMRPhenomXPHM-SpinTaylor",
    "C00:Mixed",
    "C00:SEOBNRv5PHM",
]

metadata_paths = {
    "approximant": "approximant",
    "preferred": "meta_data/other/preferred",
    "pe_algorithm": "meta_data/sampler/pe_algorithm",
    "sampler": "config_file/config/sampler",
    "cosmology": "config_file/config/cosmology",
    "default_prior": "config_file/config/default-prior",
}

def decode_value(value):
    arr = np.asarray(value).reshape(-1)
    if len(arr) != 1:
        return str(value)
    value = arr[0]
    return value.decode() if isinstance(value, (bytes, np.bytes_)) else value

metadata_rows = []
with h5py.File(pe_path, "r") as f:
    for label in labels:
        row = {"label": label}
        for name, subpath in metadata_paths.items():
            path = f"{label}/{subpath}"
            row[name] = decode_value(f[path][()]) if path in f else None
        metadata_rows.append(row)

    mixed_components = decode_value(
        f["C00:Mixed/meta_data/meta_data/combined"][()]
    )

print("C00:Mixed combines:", mixed_components)
pd.DataFrame(metadata_rows)


C00:Mixed combines: C00:IMRPhenomXPHM-SpinTaylor, C00:SEOBNRv5PHM


,label,approximant,preferred,pe_algorithm,sampler,cosmology,default_prior
0,C00:IMRPhenomXPHM-SpinTaylor,IMRPhenomXPHM,False,bilby,dynesty,Planck15_lal,BBHPriorDict
1,C00:Mixed,None,None,None,None,None,None
2,C00:SEOBNRv5PHM,SEOBNRv5PHM,False,bilby,dynesty,Planck15_lal,BBHPriorDict


## 4. PE-prior consistency across component analyses

`C00:Mixed` combines the two waveform-specific analyses. Verify that their stored analytic prior definitions are identical before treating the mixture as sharing one PE prior.


In [5]:
with h5py.File(pe_path, "r") as f:
    label1 = "C00:IMRPhenomXPHM-SpinTaylor"
    label2 = "C00:SEOBNRv5PHM"
    p1 = f[f"{label1}/priors/analytic"]
    p2 = f[f"{label2}/priors/analytic"]

    names1, names2 = set(p1.keys()), set(p2.keys())
    common = sorted(names1 & names2)
    differences = [
        name for name in common
        if not np.array_equal(p1[name][()], p2[name][()])
    ]

prior_check = {
    "only_in_imrphenom": sorted(names1 - names2),
    "only_in_seobnr": sorted(names2 - names1),
    "common_prior_entries": len(common),
    "different_prior_entries": len(differences),
}
prior_check

{'only_in_imrphenom': [],
 'only_in_seobnr': [],
 'common_prior_entries': 77,
 'different_prior_entries': 0}

## 5. Current decision record — `GW230627_015337`

**Candidate labels**
- `C00:IMRPhenomXPHM-SpinTaylor`
- `C00:Mixed`
- `C00:SEOBNRv5PHM`

**File-level findings**
- All three labels contain the required population coordinates.
- `C00:Mixed` combines the IMRPhenomXPHM-SpinTaylor and SEOBNRv5PHM analyses.
- The two component analyses use Bilby/Dynesty, `Planck15_lal`, and `BBHPriorDict`.
- Their stored analytic prior definitions match exactly: 77 common entries, 0 differences.

**Frozen selection principle**
Use the official release-designated default PE result; do not select a label using sample count, ESS, or the eventual population score.

**Selected label for this event**
`C00:Mixed`, subject to retaining the official release documentation as provenance for the default-PE designation.


## 6. Exact PE-prior reconstruction

Goal: reconstruct the PE prior in population coordinates

\[
(m_{1,\rm source},m_{2,\rm source},z)
\]

from the released sampling coordinates \((\mathcal M_{\rm det},q,d_L)\). The required transformation is

\[
\pi_{\rm PE}(m_{1,s},m_{2,s},z)
=
\pi_{\rm PE}(\mathcal M_{\rm det},q,d_L)
\left|\frac{\partial(\mathcal M_{\rm det},q,d_L)}
{\partial(m_{1,s},m_{2,s},z)}\right|.
\]

The selected `C00:Mixed` posterior shares the same analytic PE prior as its two component analyses.

### 6.1 Load posterior coordinates and exact cosmology

In [6]:
label = "C00:Mixed"

with h5py.File(pe_path, "r") as f:
    samples = f[f"{label}/posterior_samples"][()]

m1_det = samples["mass_1"]
m2_det = samples["mass_2"]
m1_src = samples["mass_1_source"]
m2_src = samples["mass_2_source"]
z = samples["redshift"]
dL = samples["luminosity_distance"]
chirp_stored = samples["chirp_mass"]
q = samples["mass_ratio"]

cosmology = get_cosmology("Planck15_lal")

print(f"samples: {len(samples)}")
print(cosmology)

samples: 32382
FlatLambdaCDM(name="Planck15_LAL", H0=67.9 km / (Mpc s), Om0=0.3065, Tcmb0=0.0 K, Neff=3.04, m_nu=None, Ob0=0.0)


/Users/924994587/.conda/envs/gwtc_forecast/lib/python3.11/site-packages/bilby/gw/cosmology.py:64: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import PC_SI as LAL_PC_SI


### 6.2 Coordinate helpers

The released `chirp_mass` is verified below to be detector-frame. For source-frame component masses,

\[
\mathcal M_{\rm det}=(1+z)\mathcal M_{\rm source},
\qquad q=\frac{m_2}{m_1}.
\]

For the flat `Planck15_LAL` cosmology,

\[
\frac{dd_L}{dz}=D_C(z)+(1+z)\frac{c}{H(z)}.
\]

Therefore

\[
J=\left|\frac{\partial(\mathcal M_{\rm det},q,d_L)}
{\partial(m_{1,s},m_{2,s},z)}\right|
=
\frac{(1+z)q^{3/5}}{m_{1,s}(1+q)^{1/5}}
\frac{dd_L}{dz}.
\]

In [7]:
def chirp_mass(m1, m2):
    return (m1 * m2)**(3 / 5) / (m1 + m2)**(1 / 5)


def ddL_dz(z, cosmology):
    Dc = cosmology.comoving_distance(z)
    return (Dc + (1 + z) * c / cosmology.H(z)).to("Mpc").value


def pe_coordinate_jacobian(m1_source, m2_source, z, cosmology):
    q = m2_source / m1_source
    mass_factor = (
        (1 + z)
        * q**(3 / 5)
        / (m1_source * (1 + q)**(1 / 5))
    )
    return mass_factor * ddL_dz(z, cosmology)

### 6.3 Validate the coordinate transformation

In [8]:
chirp_from_det = chirp_mass(m1_det, m2_det)
chirp_from_src = chirp_mass(m1_src, m2_src)

assert np.allclose(chirp_stored, chirp_from_det)
assert not np.allclose(chirp_stored, chirp_from_src)
assert np.allclose(chirp_stored, (1 + z) * chirp_from_src)

J = pe_coordinate_jacobian(m1_src, m2_src, z, cosmology)
J_alt = chirp_stored / m1_src**2 * ddL_dz(z, cosmology)

assert np.all(np.isfinite(J))
assert np.all(J > 0)
assert np.allclose(J, J_alt)

print("chirp_mass is detector-frame: verified")
print("Jacobian finite and positive: verified")
print("Equivalent Jacobian forms agree: verified")
print(f"J range: {J.min():.6f} -- {J.max():.6f}")

chirp_mass is detector-frame: verified
Jacobian finite and positive: verified
Equivalent Jacobian forms agree: verified
J range: 155.122931 -- 685.862017


### 6.4 Reconstruct the released mass-distance prior

The stored analytic priors are:

- `UniformInComponentsChirpMass`: \(p(\mathcal M)\propto\mathcal M\)
- `UniformInComponentsMassRatio`: \(p(q)\propto(1+q)^{2/5}/q^{6/5}\)
- `UniformSourceFrame`: \(p(z)\propto (dV_c/dz)/(1+z)\), represented in \(d_L\)
- `mass_1` and `mass_2`: support constraints only, not independent density factors.

Use Bilby's own prior classes so normalization and support match the released PE analysis.

In [9]:
chirp_prior = UniformInComponentsChirpMass(
    minimum=6.38537095164235,
    maximum=6.4428834107576165,
    name="chirp_mass",
)

q_prior = UniformInComponentsMassRatio(
    minimum=0.05,
    maximum=1.0,
    name="mass_ratio",
)

distance_prior = UniformSourceFrame(
    minimum=50.0,
    maximum=471.599,
    cosmology=cosmology,
    name="luminosity_distance",
)

mass_support = (
    (m1_det >= 1) & (m1_det <= 1000)
    & (m2_det >= 1) & (m2_det <= 1000)
)

log_p_chirp = chirp_prior.ln_prob(chirp_stored)
log_p_q = q_prior.ln_prob(q)
log_p_dL = distance_prior.ln_prob(dL)

assert np.all(np.isfinite(log_p_chirp))
assert np.all(np.isfinite(log_p_q))
assert np.all(np.isfinite(log_p_dL))
assert np.all(mass_support)

### 6.5 Validation checks for the released prior

In [10]:
# Mass check: the paired Bilby priors should become uniform in
# detector-frame component masses after the mass-coordinate Jacobian.
J_mass_det = q**(3 / 5) / (m1_det * (1 + q)**(1 / 5))
log_mass_transformed = log_p_chirp + log_p_q + np.log(J_mass_det)

# Distance check: p(dL) * |ddL/dz| should reproduce the UniformSourceFrame
# redshift shape up to a normalization constant and Bilby's interpolation error.
distance_transformed = log_p_dL + np.log(ddL_dz(z, cosmology))
log_pz_shape = (
    np.log(cosmology.differential_comoving_volume(z).value)
    - np.log1p(z)
)
distance_offset = distance_transformed - log_pz_shape

print("mass transformed range:", np.ptp(log_mass_transformed))
print("distance offset std:", np.std(distance_offset))
print("distance offset range:", np.ptp(distance_offset))

assert np.ptp(log_mass_transformed) < 1e-12
assert np.ptp(distance_offset) < 2e-3

mass transformed range: 2.220446049250313e-15
distance offset std: 0.0001619883208584841
distance offset range: 0.0010166218525000659


### 6.6 Final mass-redshift PE prior

In the population coordinates,

\[
\log\pi_{\rm PE}(m_{1,s},m_{2,s},z)
=
\log p(\mathcal M_{\rm det})
+
\log p(q)
+
\log p(d_L)
+
\log J,
\]

with the released detector-frame component-mass constraints enforced.

In [11]:
log_pe_prior_mass_z = (
    log_p_chirp
    + log_p_q
    + log_p_dL
    + np.log(J)
)
log_pe_prior_mass_z[~mass_support] = -np.inf

finite = np.isfinite(log_pe_prior_mass_z)
assert np.all(finite)

print("All finite on support:", np.all(finite))
print("min:", np.min(log_pe_prior_mass_z))
print("median:", np.median(log_pe_prior_mass_z))
print("max:", np.max(log_pe_prior_mass_z))

All finite on support: True
min: -1.2549831366592965
median: 2.2198031074357782
max: 2.854056480127734


## 7. Status and next step

Validated for `GW230627_015337`:

- selected PE label and mixed-posterior provenance;
- identical analytic priors for both component analyses;
- detector-frame interpretation of `chirp_mass`;
- exact `Planck15_LAL` cosmology;
- analytic luminosity-distance derivative;
- full source-frame mass/redshift Jacobian;
- mass-prior transformation to uniform detector-frame component masses;
- `UniformSourceFrame` distance transformation;
- reconstructed `log_pe_prior_mass_z` on all posterior samples.

**Next:** inspect the frozen GWTC-3 population-model interface to determine the required spin coordinates. Add only the spin-prior terms (and any necessary spin-coordinate Jacobian) before defining the final `EventPosterior` contract.